# 별의 끝에서 — 별빛 세계 배경 이미지 생성 (교내 GPU 틸론)

한끗차이 웹사이트의 게임 《별의 끝에서》 지역 배경을 생성합니다.
**순서대로 위에서부터 실행**하세요. 결과는 `outputs/` 폴더와 `starlight_outputs.zip`에 저장됩니다.

| 단계 | 내용 |
|---|---|
| 1 | GPU·메모리·Python·CUDA 확인 → `outputs/gpu_env.json` |
| 2 | 라이브러리 설치 |
| 3 | GPU 메모리에 맞춰 모델 자동 선택 |
| 4 | 6장 생성 + 생성 기록 `outputs/generation_log.json` |
| 5 | 웹용 WebP 변환 + zip 압축 (이 파일을 내려받아 프로젝트에 넣으면 됩니다) |

In [ ]:
# 1. 환경 확인: GPU 종류, 메모리, Python, CUDA
import json, platform, subprocess, sys, datetime, os
os.makedirs("outputs", exist_ok=True)
env = {"checked_at": datetime.datetime.now().isoformat(timespec="seconds"), "python": sys.version.split()[0], "platform": platform.platform()}
try:
    smi = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader"], capture_output=True, text=True, timeout=30)
    env["nvidia_smi"] = smi.stdout.strip() or smi.stderr.strip()
except Exception as e:
    env["nvidia_smi"] = f"실행 불가: {e}"
try:
    import torch
    env["torch"] = torch.__version__
    env["cuda_available"] = torch.cuda.is_available()
    env["cuda_version"] = torch.version.cuda
    if torch.cuda.is_available():
        p = torch.cuda.get_device_properties(0)
        env["gpu_name"] = p.name
        env["gpu_memory_gb"] = round(p.total_memory / 1024**3, 1)
except ImportError:
    env["torch"] = "미설치 (2단계에서 설치)"
json.dump(env, open("outputs/gpu_env.json", "w", encoding="utf-8"), ensure_ascii=False, indent=2)
print(json.dumps(env, ensure_ascii=False, indent=2))

In [ ]:
# 2. 라이브러리 설치 (인터넷 연결 필요. 이미 있으면 빠르게 넘어갑니다)
%pip install -q "diffusers>=0.30" "transformers>=4.44" accelerate safetensors pillow
# torch가 없거나 CUDA를 못 쓰면 아래 주석을 풀고 실행 (CUDA 12.1 예시)
# %pip install -q torch --index-url https://download.pytorch.org/whl/cu121

In [ ]:
# 3. GPU 메모리에 맞춰 모델 선택
import torch
assert torch.cuda.is_available(), "CUDA GPU를 찾지 못했어요. 1단계 결과를 확인해 주세요."
vram = torch.cuda.get_device_properties(0).total_memory / 1024**3
if vram >= 12:
    MODEL = "stabilityai/stable-diffusion-xl-base-1.0"   # 라이선스: CreativeML Open RAIL++-M
    SIZE, STEPS, GUIDANCE = (1344, 768), 30, 6.5
else:
    MODEL = "stable-diffusion-v1-5/stable-diffusion-v1-5"  # 라이선스: CreativeML Open RAIL-M
    SIZE, STEPS, GUIDANCE = (912, 512), 30, 7.0
from diffusers import AutoPipelineForText2Image
pipe = AutoPipelineForText2Image.from_pretrained(MODEL, torch_dtype=torch.float16, variant="fp16", use_safetensors=True).to("cuda")
pipe.enable_attention_slicing()
print(f"GPU {torch.cuda.get_device_name(0)} ({vram:.1f}GB) → 모델 {MODEL}, 크기 {SIZE}, {STEPS} steps")

In [ ]:
# 4. 별빛 세계 이미지 생성 (사람·글자 없이 배경만)
import time, json
STYLE = "soft watercolor illustration, 2.5D game background, dreamy starlight world, pastel purple, deep navy and warm gold palette, glowing stars, gentle light, wide landscape, no people, no text"
NEG = "people, person, character, text, letters, watermark, logo, signature, blurry, lowres, photo, harsh, dark horror"
PROMPTS = {
    "title":  "a vast night sky full of stars over a quiet horizon, a single bright star at the end of the sky",
    "house":  "inside a cozy old star house, tall arched windows showing the night sky, wooden bookshelves, hanging star lanterns",
    "garden": "a forgotten garden at dusk, weeping willow trees, flower arches, glowing flowers, soft mist",
    "plain":  "a wide windy plain under twilight, drifting soft shadows, scattered points of light on the grass",
    "attic":  "an attic of time, old clocks, unopened boxes, an old map and a tall mirror, warm dust in moonlight",
    "hall":   "a grand hall of stars, constellations glowing on a domed ceiling, light gathering at an altar",
}
log = {"model": MODEL, "gpu": torch.cuda.get_device_name(0), "size": SIZE, "steps": STEPS, "guidance": GUIDANCE, "style": STYLE, "negative": NEG, "images": []}
os.makedirs("outputs/raw", exist_ok=True)
torch.cuda.reset_peak_memory_stats()
for i, (name, prompt) in enumerate(PROMPTS.items()):
    seed = 2026 + i
    t0 = time.time()
    img = pipe(prompt=f"{prompt}, {STYLE}", negative_prompt=NEG, width=SIZE[0], height=SIZE[1], num_inference_steps=STEPS, guidance_scale=GUIDANCE, generator=torch.Generator("cuda").manual_seed(seed)).images[0]
    sec = round(time.time() - t0, 1)
    img.save(f"outputs/raw/{name}.png")
    log["images"].append({"name": name, "prompt": prompt, "seed": seed, "seconds": sec})
    print(f"{name}: {sec}s")
log["peak_gpu_memory_gb"] = round(torch.cuda.max_memory_allocated() / 1024**3, 2)
log["finished_at"] = datetime.datetime.now().isoformat(timespec="seconds")
log["nvidia_smi_after"] = subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout
json.dump(log, open("outputs/generation_log.json", "w", encoding="utf-8"), ensure_ascii=False, indent=2)
print("완료, 최대 GPU 메모리", log["peak_gpu_memory_gb"], "GB")

In [ ]:
# 5. 웹용 WebP(1600x900) 변환 + 미리보기 + zip
from PIL import Image
import shutil
os.makedirs("outputs/web", exist_ok=True)
for name in PROMPTS:
    im = Image.open(f"outputs/raw/{name}.png").convert("RGB")
    w, h = im.size; target = 16 / 9
    if w / h > target: nw = int(h * target); im = im.crop(((w - nw) // 2, 0, (w - nw) // 2 + nw, h))
    else: nh = int(w / target); im = im.crop((0, (h - nh) // 2, w, (h - nh) // 2 + nh))
    im.resize((1600, 900), Image.LANCZOS).save(f"outputs/web/{name}.webp", "WEBP", quality=82)
shutil.make_archive("starlight_outputs", "zip", "outputs")
print("저장 완료 → starlight_outputs.zip 을 내려받아 HanKeut/tilon-ai/ 폴더에 넣어 주세요.")
from IPython.display import display
for name in PROMPTS: display(Image.open(f"outputs/web/{name}.webp").resize((480, 270)))